# Telecom-T2C-Trainer

Continue-LoRA (QLoRA) fine-tuning of `google/gemma-4-12B-it` on the telecom NL -> TIR (PASS_0-4) multi-turn dataset.

**This notebook only orchestrates** — all logic lives in `src/`. The only file you should ever need to edit between runs is `configs/experiment.yaml`.

Run cells top to bottom. Sections: Runtime Check, Install, Configuration, Load Dataset, Dataset Statistics, Token Analysis, Load Model, Load Adapter, Train, Evaluate, Save, Smoke Test.

## 1. Runtime Check

In [ ]:
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Locate the Telecom-T2C project root from any Colab/local starting cwd."""
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for child in start.glob("*/"):
        if (child / "src").is_dir() and (child / "configs").is_dir():
            return child
    raise RuntimeError(
        "Could not locate the Telecom-T2C project root (a directory containing both "
        "'src/' and 'configs/'). If running in Colab, cd into the cloned/uploaded repo "
        "directory first, e.g.:\n  %cd /content/Telecom-T2C"
    )


PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src import utils

logger = utils.setup_logging()
gpu = utils.detect_gpu()
print(f"GPU: {gpu.name} | family={gpu.family} | vram={gpu.vram_gb:.1f} GB | bf16={gpu.bf16_supported}")

if gpu.family == "CPU":
    raise RuntimeError(
        "No GPU detected. In Colab: Runtime -> Change runtime type -> select a GPU "
        "(A100 recommended for this 12B model)."
    )
elif gpu.family != "A100":
    print(
        f"Warning: this notebook's defaults are tuned for A100. Detected {gpu.family} — "
        "Section 7 (Load Model) will print recommended overrides for configs/experiment.yaml."
    )

## 2. Install

Re-run this cell after any runtime restart, and again any time you pull a `requirements.txt` change.

**Known Colab quirks** (both self-diagnosed by the version-check cell below):
- `numpy.dtype size changed, may indicate binary incompatibility` — pip reconciled numpy/pandas/pyarrow versions inside this already-running kernel (the old binaries were compiled against Colab's pre-installed numpy).
- `Could not import module 'X'` from `peft` or `trl` — the installed peft/trl release predates the transformers version Colab already has (transformers is intentionally left unpinned-above in `requirements.txt` to support new base models; peft/trl float with it for the same reason).

Either way: **re-run this Install section's pip-install cell**, then **Runtime -> Restart session**, then re-run the notebook from Section 1.

In [ ]:
import subprocess, sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")]
)

In [ ]:
import importlib

_RESTART_HINT = (
    "Re-run this Install section's pip-install cell (in case requirements.txt "
    "changed or Colab's environment drifted since your last run), then "
    "Runtime -> Restart session, then re-run the notebook from Section 1. "
    "A numpy/pandas 'binary incompatibility' error, or a peft/trl "
    "'Could not import module' error, both mean package versions were "
    "reconciled inside this already-running kernel and it needs a fresh process."
)

failures = []
for pkg in ("transformers", "datasets", "peft", "trl", "bitsandbytes", "accelerate"):
    try:
        module = importlib.import_module(pkg)
        print(f"{pkg}: {getattr(module, '__version__', 'ok')}")
    except Exception as exc:
        print(f"{pkg}: FAILED — {exc}")
        failures.append(pkg)

if failures:
    print(f"\n{len(failures)} package(s) failed to import: {', '.join(failures)}")
    print(_RESTART_HINT)

## 3. Configuration

Loads `configs/experiment.yaml` — edit that file, not this cell, to change settings.

In [ ]:
from src import config as config_mod
from src import manifest as manifest_mod

CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment.yaml"
experiment_config = config_mod.load_config(CONFIG_PATH)

for warning in config_mod.validate_config(experiment_config):
    print(f"WARNING: {warning}")

prior_manifest = manifest_mod.load_prior_manifest(experiment_config.model.continue_adapter)
experiment_config = manifest_mod.apply_prior_manifest_inheritance(experiment_config, prior_manifest)

utils.set_seed(experiment_config.identity.seed)

if experiment_config.drive.google_drive_directory:
    utils.mount_google_drive(experiment_config.drive.drive_mount_point)

run_dir = config_mod.resolve_run_dir(experiment_config)
config_mod.save_resolved_config(experiment_config, run_dir / "config.yaml")
print(f"Run directory: {run_dir}")
print(f"Base model: {experiment_config.model.base_model}")
print(f"Continue adapter: {experiment_config.model.continue_adapter or '(none — fresh LoRA init)'}")

## 4. Load Dataset

In [ ]:
from src import tokenizer as tokenizer_mod
from src import dataset as dataset_mod

hf_token = tokenizer_mod.resolve_hf_token(experiment_config.model.hf_token_env_var)
tokenizer = tokenizer_mod.load_tokenizer(experiment_config.model.base_model, hf_token)

loader = dataset_mod.DatasetLoader(experiment_config.data, tokenizer, experiment_config.identity.seed)
train_ds = loader.load_train()
val_ds = loader.load_validation()
golden_ds = loader.load_golden()

print(
    f"train={len(train_ds):,}  "
    f"val={len(val_ds):,}" if val_ds is not None else "val=0",
    f"golden={len(golden_ds):,}" if golden_ds is not None else "golden=0 (skipped)",
)

## 5. Dataset Statistics

Tokenizes each split once; the results are reused by Section 6 (Token Analysis) rather than re-tokenizing — the train split alone is ~91k conversations.

In [ ]:
from src import statistics as statistics_mod
from src import model as model_mod

gpu_profile = model_mod.detect_gpu_profile(experiment_config.hardware.gpu_profile_override)
if gpu_profile.family != "A100":
    print(f"GPU profile note: {gpu_profile.notes}")
    print(
        f"  recommended batch_size={gpu_profile.recommended_batch_size}, "
        f"gradient_accumulation={gpu_profile.recommended_gradient_accumulation}, "
        f"max_train_samples={gpu_profile.recommended_max_train_samples}"
    )

token_analyses = {}
for split_name, split_ds in (("train", train_ds), ("val", val_ds), ("golden", golden_ds)):
    if split_ds is None:
        continue
    stats = statistics_mod.compute_dataset_statistics(
        split_ds, tokenizer, experiment_config.data.max_seq_length, split_name,
        experiment_config.training.batch_size, experiment_config.training.gradient_accumulation,
        experiment_config.training.epochs, experiment_config.training.packing, gpu_profile.family,
    )
    statistics_mod.print_statistics_report(stats)
    token_analyses[split_name] = statistics_mod.analyze_tokens(
        split_ds, tokenizer, experiment_config.data.max_seq_length
    )
    statistics_mod.display_histogram(
        token_analyses[split_name].per_example_lengths,
        title=f"\nToken-length histogram: {split_name}",
    )

## 6. Token Analysis

Reuses the tokenization already computed in Section 5.

In [ ]:
for split_name, analysis in token_analyses.items():
    print(f"=== Token analysis: {split_name} ===")
    print(f"  mean:   {analysis.mean:.1f}")
    print(f"  median: {analysis.median:.1f}")
    print(f"  p95:    {analysis.p95:.1f}")
    print(f"  max:    {analysis.max}")
    print(
        f"  exceeding max_seq_length ({experiment_config.data.max_seq_length}): "
        f"{analysis.num_exceeding_max_seq_length}"
    )

## 7. Load Model

4-bit QLoRA base model load. This is the step most likely to need iteration on a brand-new base model — see README "Troubleshooting" if it fails with an unrecognized-architecture error.

In [ ]:
model_mod.configure_cuda_visible_devices(experiment_config.hardware.training_gpu)
base_model = model_mod.load_base_model(
    experiment_config.model, hf_token, device_index=experiment_config.hardware.training_gpu
)

## 8. Load Adapter

Continues `model.continue_adapter` if set in the config, otherwise initializes a fresh LoRA adapter. Never merges the adapter into the base model.

In [ ]:
peft_model = model_mod.attach_lora(
    base_model, experiment_config.lora, experiment_config.model.continue_adapter
)

## 9. Train

Long-running cell. Automatically resumes from the latest checkpoint under this run's `adapter/` directory if `reproducibility.resume_training` is true and a prior interrupted run is found.

In [ ]:
from src import checkpoint as checkpoint_mod
from src import wandb_logger as wandb_logger_mod
from src import trainer as trainer_mod

run_metadata = {
    "experiment_name": experiment_config.identity.experiment_name,
    "dataset_version": experiment_config.identity.dataset_version,
    "lora_version": experiment_config.identity.lora_version,
    "generator_version": experiment_config.identity.generator_version,
    "validator_version": experiment_config.identity.validator_version,
    "git_hash": utils.get_git_hash(),
    "base_model": experiment_config.model.base_model,
}

wb_logger = wandb_logger_mod.WandbLogger(experiment_config.wandb, run_metadata)
wb_logger.init()

eval_dataset_for_training = (
    golden_ds if experiment_config.data.eval_source == "golden" and golden_ds is not None else val_ds
)

sft_trainer = trainer_mod.train(
    experiment_config, peft_model, tokenizer, train_ds, eval_dataset_for_training,
    run_dir, wb_logger, run_metadata,
)
trainer_mod.save_best_model(sft_trainer, run_dir / "adapter", tokenizer)

## 10. Evaluate

Loss-based validation metrics, plus a golden generation-eval + benchmark report if `data.golden_path` is configured (skipped gracefully otherwise).

In [ ]:
from src import evaluator as evaluator_mod
from src import benchmark as benchmark_mod
from src import inference as inference_mod

val_metrics = None
if eval_dataset_for_training is not None:
    val_metrics = evaluator_mod.evaluate_validation(sft_trainer)
    print("Validation metrics:", val_metrics)

report = benchmark_mod.run_benchmark(
    experiment_config, run_dir / "adapter", run_dir, golden_ds,
    hf_token=hf_token, val_metrics=val_metrics,
)
benchmark_mod.write_benchmark_report(report, run_dir / "metrics" / "benchmark_report.json")
print("Golden metrics:", report.golden_metrics)
print("PASS metric status (interfaces only, not yet implemented):", report.pass_metric_status)

## 11. Save

Writes `manifest.json`, zips the adapter, syncs the run to Google Drive, and uploads artifacts to wandb (each step no-ops gracefully if Drive/wandb aren't available).

In [ ]:
final_manifest = manifest_mod.build_manifest(
    experiment_config, prior_manifest, len(train_ds),
    len(val_ds) if val_ds is not None else 0,
    len(golden_ds) if golden_ds is not None else None,
    run_dir.name,
)
manifest_mod.write_manifest(final_manifest, run_dir / "manifest.json")

zip_path = checkpoint_mod.zip_adapter(
    run_dir / "adapter", run_dir / f"{experiment_config.identity.experiment_name}.zip"
)
checkpoint_mod.sync_run_to_drive(
    run_dir, experiment_config.drive.google_drive_directory, experiment_config.drive.drive_mount_point
)

exp_name = experiment_config.identity.experiment_name
for artifact_path, artifact_type, artifact_name in (
    (run_dir / "manifest.json", "manifest", f"{exp_name}-manifest"),
    (run_dir / "adapter", "model", f"{exp_name}-adapter"),
    (run_dir / "predictions", "predictions", f"{exp_name}-predictions"),
    (run_dir / "metrics", "metrics", f"{exp_name}-metrics"),
    (run_dir / "config.yaml", "config", f"{exp_name}-config"),
):
    wb_logger.upload_artifact(artifact_path, artifact_type, artifact_name)

wb_logger.finish()
print(f"Run saved to {run_dir}")
print(f"Adapter zip: {zip_path}")

## 12. Smoke Test

Reloads the saved adapter fresh (as if after a runtime restart) and generates from one training example's prompt turns, for a quick sanity check against the gold reply.

In [ ]:
inf_model, inf_tokenizer = inference_mod.load_model_for_inference(
    experiment_config.model, str(run_dir / "adapter"), hf_token
)

sample_messages = train_ds[0]["messages"]
first_assistant_idx = next(i for i, m in enumerate(sample_messages) if m["role"] == "assistant")
prompt_messages = sample_messages[:first_assistant_idx]
gold = sample_messages[first_assistant_idx]["content"]

generated = inference_mod.run_smoke_test(
    inf_model, inf_tokenizer, prompt_messages,
    max_new_tokens=experiment_config.evaluation.max_new_tokens_eval,
)
print("\n=== GOLD ===")
print(gold[:1500])